# **Q1. Load Fashion-MNIST**

**Understand the Concept**

- Fashion-MNIST contains grayscale clothing images.
- It has 10 classes and is beginner-friendly for CNNs.

In [ ]:
import torch
from torchvision import datasets, transforms

transform = transforms.ToTensor()
train_dataset = datasets.FashionMNIST("./data", train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST("./data", train=False, download=True, transform=transform)
print(len(train_dataset), len(test_dataset))

## **Explanation**

- `ToTensor()` converts images to PyTorch tensors.

# **Q2. Visualization**

**Understand the Concept**

- Visualizing samples helps us understand images and labels.

In [ ]:
import matplotlib.pyplot as plt
image, label = train_dataset[0]
plt.imshow(image.squeeze(), cmap="gray")
plt.title(f"Label: {label}")
plt.axis("off")
plt.show()

## **Explanation**

- The image is 28×28 and grayscale.
- The label is its numeric class.

# **Q3. Normalization**

**Understand the Concept**

- Normalization can make CNN training more stable.
- Here pixels are converted to tensors and normalized around zero.

In [ ]:
from torchvision import transforms
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
print(transform)

## **Explanation**

- The transform first converts pixels to tensors and then normalizes them.

# **Q4. Train/Test Split**

**Understand the Concept**

- Fashion-MNIST already provides separate training and test sets.
- Training data learns the model; test data is used for final evaluation.

In [ ]:
print("Train:", len(train_dataset))
print("Test:", len(test_dataset))

## **Explanation**

- The two sets are kept separate during training.

# **Q5. DataLoader**

**Understand the Concept**

- DataLoader creates mini-batches for CNN training.

In [ ]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
images, labels = next(iter(train_loader))
print(images.shape, labels.shape)

## **Explanation**

- Each batch contains 64 images.
- Training data is shuffled.

# **Q6. CNN Architecture**

**Understand the Concept**

- The CNN uses convolution, ReLU, pooling, flattening, and fully connected layers.
- The final layer produces 10 class scores.

In [ ]:
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64*7*7,128), nn.ReLU(), nn.Linear(128,10)
        )
    def forward(self,x):
        return self.classifier(self.features(x))

model = CNN()
print(model)

## **Explanation**

- The CNN learns progressively deeper image features.
- The final layer has 10 outputs for the 10 classes.

# **Q7. Forward Pass**

**Understand the Concept**

- The forward pass sends a batch of images through the CNN.

In [ ]:
images, labels = next(iter(train_loader))
outputs = model(images)
print("Input:", images.shape)
print("Output:", outputs.shape)

## **Explanation**

- Each image receives 10 class scores.

# **Q8. Training**

**Understand the Concept**

- Cross Entropy Loss is used for multiclass classification.
- Adam updates CNN parameters.

In [ ]:
import torch
import torch.nn as nn
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
model.train()
for images, labels in train_loader:
    outputs = model(images)
    loss = loss_fn(outputs, labels)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
print("One epoch completed")

## **Explanation**

- The CNN learns from each mini-batch.
- The code demonstrates one training epoch.

# **Q9. Validation**

**Understand the Concept**

- Validation checks predictions without changing parameters.

In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for images, labels in test_loader:
        pred = model(images).argmax(1)
        total += labels.size(0)
        correct += (pred == labels).sum().item()
print("Accuracy:", correct/total)

## **Explanation**

- `eval()` switches to evaluation behavior.
- `no_grad()` avoids unnecessary gradient calculations.

# **Q10. Prediction**

**Understand the Concept**

- The class with the highest output score is the prediction.

In [ ]:
model.eval()
image, true_label = test_dataset[0]
with torch.no_grad():
    predicted = model(image.unsqueeze(0)).argmax(1).item()
print("True:", true_label)
print("Predicted:", predicted)

## **Explanation**

- One image is passed through the CNN and the highest-scoring class is selected.

# **Q11. Accuracy**

**Understand the Concept**

- Accuracy is the percentage of correctly classified images.

In [ ]:
from sklearn.metrics import accuracy_score
all_true, all_pred = [], []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        pred = model(images).argmax(1)
        all_true.extend(labels.numpy()); all_pred.extend(pred.numpy())
print("Accuracy:", accuracy_score(all_true, all_pred))

## **Explanation**

- Accuracy gives an overall performance score.

# **Q12. Confusion Matrix**

**Understand the Concept**

- A confusion matrix shows which classes are confused with each other.

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
cm = confusion_matrix(all_true, all_pred)
plt.imshow(cm); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix"); plt.colorbar(); plt.show()
print(cm)

## **Explanation**

- Diagonal values are correct predictions.
- Off-diagonal values are errors.

# **Q13. Per-Class Performance**

**Understand the Concept**

- Class-wise accuracy shows which classes are easy or difficult.

In [ ]:
import numpy as np
cm = confusion_matrix(all_true, all_pred)
scores = cm.diagonal()/cm.sum(axis=1)
for i, score in enumerate(scores):
    print(f"Class {i}: {score:.3f}")

## **Explanation**

- A low class score identifies a class that needs investigation.

# **Q14. Misclassified Images**

**Understand the Concept**

- Incorrect predictions are useful for error analysis.
- They show exactly where the CNN is confused.

In [ ]:
shown = 0
for images, labels in test_loader:
    with torch.no_grad(): pred = model(images).argmax(1)
    for i in range(len(labels)):
        if pred[i] != labels[i]:
            import matplotlib.pyplot as plt
            plt.imshow(images[i].squeeze(), cmap="gray")
            plt.title(f"True: {labels[i].item()} | Pred: {pred[i].item()}")
            plt.axis("off"); plt.show()
            shown += 1
            if shown == 5: break
    if shown == 5: break

## **Explanation**

- Comparing true and predicted labels helps identify common error patterns.

# **Q15. Correct Predictions**

**Understand the Concept**

- Correct predictions can also be displayed to understand what the model recognizes well.

In [ ]:
shown = 0
for images, labels in test_loader:
    with torch.no_grad(): pred = model(images).argmax(1)
    for i in range(len(labels)):
        if pred[i] == labels[i]:
            import matplotlib.pyplot as plt
            plt.imshow(images[i].squeeze(), cmap="gray")
            plt.title(f"Correct: {pred[i].item()}")
            plt.axis("off"); plt.show()
            shown += 1
            if shown == 5: break
    if shown == 5: break

## **Explanation**

- Correct examples show patterns the CNN learned successfully.